# Sentiment Analysis of Internship / Employee Feedback

### Internship Task
**Objective:** Analyze feedback to identify **Positive, Negative, and Neutral** sentiments and identify areas where satisfaction can be improved.

### Dataset
`employee_reviews.csv.csv`

### Model
**TF-IDF + Logistic Regression**

> **Important:** The dataset does not contain a ready-made sentiment column. Therefore, the `Overall_rating` column is used only to create training labels:
> - **1–2 → Negative**
> - **3 → Neutral**
> - **4–5 → Positive**
>
> The actual model input is the written feedback (`Likes` + `Dislikes`), so the rating is **not used as an input feature**.


In [ ]:
# Cell 1 — Install / import required libraries

!pip -q install scikit-learn pandas matplotlib seaborn joblib

import os
import re
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

warnings.filterwarnings("ignore")

print("Libraries loaded successfully.")


In [ ]:
# Cell 2 — Upload the dataset

uploaded = files.upload()

csv_files = [name for name in uploaded.keys() if name.lower().endswith(".csv")]

if not csv_files:
    raise FileNotFoundError("No CSV file was uploaded.")

file_name = csv_files[0]
print("Using dataset:", file_name)


In [ ]:
# Cell 3 — Load and inspect the dataset

df = pd.read_csv(file_name)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

display(df.head())

print("\nMissing values:")
display(df.isnull().sum())


## 1. Data Preparation

We will combine the two textual feedback fields:

- `Likes`
- `Dislikes`

The model will learn sentiment from this text.

Rows without `Overall_rating` cannot be used for supervised training because they do not have a target label.


In [ ]:
# Cell 4 — Create the 3 sentiment classes

# Keep only rows with a valid overall rating
data = df[df["Overall_rating"].isin([1, 2, 3, 4, 5])].copy()

# Combine written feedback
data["Feedback"] = (
    data["Likes"].fillna("").astype(str) + " " +
    data["Dislikes"].fillna("").astype(str)
)

# Remove empty feedback rows
data = data[data["Feedback"].str.strip().ne("")].copy()

# Convert rating to 3 sentiment classes
rating_to_sentiment = {
    1: "Negative",
    2: "Negative",
    3: "Neutral",
    4: "Positive",
    5: "Positive"
}

data["Sentiment"] = data["Overall_rating"].map(rating_to_sentiment)

# Basic text cleaning
def clean_text(text):
    text = str(text)
    text = re.sub(r"Read More", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"\\s+", " ", text)
    return text.strip()

data["Feedback"] = data["Feedback"].apply(clean_text)

print("Training rows:", len(data))
print("\nSentiment distribution:")
display(data["Sentiment"].value_counts())

display(data[["Overall_rating", "Sentiment", "Feedback"]].head())


In [ ]:
# Cell 5 — Visualize sentiment distribution

plt.figure(figsize=(7, 5))
sns.countplot(data=data, x="Sentiment", order=["Negative", "Neutral", "Positive"])
plt.title("Sentiment Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Number of Reviews")
plt.show()


## 2. Train / Test Split

We use an **80/20 stratified split** so that all three sentiment classes remain represented in both training and testing data.


In [ ]:
# Cell 6 — Train/test split

X = data["Feedback"]
y = data["Sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))


## 3. Build the Sentiment Model

### TF-IDF
Converts written feedback into numerical features based on important words and phrases.

### Logistic Regression
Learns the relationship between those text features and the three classes:

**Negative | Neutral | Positive**

`class_weight="balanced"` is used because the dataset contains more positive reviews than neutral/negative reviews.


In [ ]:
# Cell 7 — Build and train TF-IDF + Logistic Regression

sentiment_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            ngram_range=(1, 2),
            min_df=2,
            max_features=120000,
            sublinear_tf=True
        )
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1500,
            class_weight="balanced",
            random_state=42
        )
    )
])

sentiment_model.fit(X_train, y_train)

print("Model training completed successfully.")


In [ ]:
# Cell 8 — Evaluate the model

y_pred = sentiment_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")

print(f"Accuracy : {accuracy:.4f}")
print(f"Macro F1 : {macro_f1:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        labels=["Negative", "Neutral", "Positive"],
        digits=4
    )
)


In [ ]:
# Cell 9 — Confusion matrix

labels = ["Negative", "Neutral", "Positive"]

cm = confusion_matrix(y_test, y_pred, labels=labels)

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=labels,
    yticklabels=labels
)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Sentiment")
plt.ylabel("Actual Sentiment")
plt.show()


## 4. Interactive Sentiment Prediction

This section allows a user to enter any internship/employee feedback and receive:

- **Positive**
- **Negative**
- **Neutral**

It also displays the model's probability estimate for each class.


In [ ]:
# Cell 10 — Single interactive prediction

def predict_sentiment(feedback):
    feedback = clean_text(feedback)

    if not feedback:
        print("Please enter some feedback.")
        return

    prediction = sentiment_model.predict([feedback])[0]
    probabilities = sentiment_model.predict_proba([feedback])[0]
    classes = sentiment_model.named_steps["classifier"].classes_

    probability_table = (
        pd.DataFrame({
            "Sentiment": classes,
            "Probability": probabilities
        })
        .sort_values("Probability", ascending=False)
        .reset_index(drop=True)
    )

    print("\nPredicted Sentiment:", prediction)
    print(f"Confidence: {probabilities.max() * 100:.2f}%")
    print("\nClass probabilities:")
    display(probability_table)

# Enter your own feedback here
user_feedback = input("Enter internship/employee feedback: ")
predict_sentiment(user_feedback)


In [ ]:
# Cell 11 — Continuous user interaction

print("Interactive Sentiment Analyzer")
print("Type 'exit' to stop.\n")

while True:
    user_feedback = input("Enter feedback: ")

    if user_feedback.strip().lower() == "exit":
        print("Sentiment analyzer closed.")
        break

    predict_sentiment(user_feedback)
    print("\n" + "-" * 60 + "\n")


## 5. Identify Areas Where Satisfaction Can Be Improved

For the internship requirement, we can inspect negative feedback and look for recurring improvement areas.

The following keyword groups are used only for **interpretable analysis**; they are not used as model features.


In [ ]:
# Cell 12 — Analyze common improvement areas in negative feedback

negative_text = data.loc[
    data["Sentiment"] == "Negative",
    "Feedback"
].str.lower()

improvement_categories = {
    "Salary & Benefits": [
        "salary", "pay", "bonus", "increment", "benefit",
        "compensation", "hike"
    ],
    "Management": [
        "manager", "management", "leadership", "boss",
        "micromanagement", "micro management"
    ],
    "Work-Life Balance": [
        "work life", "work-life", "weekend", "overtime",
        "workload", "extra hours"
    ],
    "Career Growth": [
        "growth", "promotion", "promoted", "career",
        "opportunity"
    ],
    "Job Security": [
        "job security", "layoff", "laid off", "fire", "fired",
        "bench"
    ],
    "Work Environment": [
        "culture", "toxic", "environment", "politics",
        "harassment"
    ],
    "Learning & Development": [
        "learning", "training", "skill", "technology",
        "course", "development"
    ]
}

area_counts = {}

for category, keywords in improvement_categories.items():
    count = 0
    for keyword in keywords:
        count += negative_text.str.contains(
            re.escape(keyword),
            regex=True,
            na=False
        ).sum()
    area_counts[category] = int(count)

areas_df = (
    pd.DataFrame(
        list(area_counts.items()),
        columns=["Improvement Area", "Keyword Mentions"]
    )
    .sort_values("Keyword Mentions", ascending=False)
    .reset_index(drop=True)
)

display(areas_df)

plt.figure(figsize=(9, 5))
sns.barplot(
    data=areas_df,
    x="Keyword Mentions",
    y="Improvement Area"
)
plt.title("Potential Areas for Improving Employee Satisfaction")
plt.xlabel("Keyword Mentions in Negative Feedback")
plt.ylabel("Area")
plt.show()


## 6. Save the Trained Model

The complete Pipeline contains both the TF-IDF vectorizer and Logistic Regression classifier, so it can be loaded later without rebuilding the preprocessing steps.


In [ ]:
# Cell 13 — Save model and supporting information

model_path = "internship_sentiment_model.pkl"

joblib.dump(sentiment_model, model_path)

print("Saved:", model_path)
print("File size:", round(os.path.getsize(model_path) / (1024 * 1024), 2), "MB")


In [ ]:
# Cell 14 — Download the trained model (optional)

files.download(model_path)


# Final Result

The completed project provides:

1. **Data preprocessing**
2. **Three-class sentiment labeling**
   - Negative
   - Neutral
   - Positive
3. **TF-IDF text feature extraction**
4. **Logistic Regression classification**
5. **Accuracy, Macro-F1 and classification report**
6. **Confusion matrix**
7. **Interactive user input**
8. **Probability/confidence for all three classes**
9. **Analysis of potential satisfaction-improvement areas**
10. **Saved `.pkl` model for future use**

### Suggested project conclusion

> The sentiment analysis system classifies written employee/internship feedback into positive, neutral, and negative categories using TF-IDF and Logistic Regression. The analysis can help identify recurring concerns in areas such as compensation, management, work-life balance, career growth, job security, work environment, and learning opportunities. The interactive prediction module also allows new feedback to be analyzed in real time.
